# Turn validator: dataset generation and gpt-oss-20b training, end to end

Each section is one step and can be re-run on its own. They run in order: data check → text model check → design → pilot → full dataset → split → GPU plan → training → reports → real logs.

1. Fill in **Inputs** below: the vLLM server, or the Microsoft gateway, and your GPUs.
2. Put the AOP files in `turnval/data/`: `aop_flow.json` and `hrt_graph.json`.
3. Run the sections top to bottom.

`SMOKE_TEST = True` runs everything with a stand-in vLLM server and a tiny model, with no GPU and no server. Use it to check the notebook itself.

In [ ]:
# ---------------------------------------------------------------- Inputs
import os
from pathlib import Path

REPO = str(Path.cwd().resolve().parents[1]) if Path.cwd().name == "notebooks" else str(Path.cwd().resolve())

# text model for generating the chats and the summaries
BACKEND = "local"                          # "local" = gpt-oss on vLLM | "gateway" = Microsoft GPT gateway
VLLM_URL = "http://<gpu-host>:8000/v1"     # the vLLM server's /v1 base (vllm serve openai/gpt-oss-20b --port 8000)
VLLM_MODEL = "openai/gpt-oss-20b"          # the model name the server serves
GATEWAY_MODEL = "gpt-41"                   # gateway deployment, used when BACKEND = "gateway"
WORKERS = 8                                # conversations generated in parallel
VERIFY = True                              # LLM quality gate on every generated reply

# dataset
OUT_DIR = "turnval/out"                    # rows.csv, train/val/test.csv, reports
PILOT_DIR = "turnval/out/pilot"
PILOT_LIMIT = 20                           # conversations in the pilot
REPLICATES = 3                             # copies of the covering array (~480 conversations each)
FULL_LIMIT = 0                             # 0 = every conversation of the design

# training (gpt-oss-20b + LoRA)
NUM_GPUS = 1
GPU_NAME = "H100 80GB"                     # e.g. "A100 80GB", "A100 40GB", "L40S", "H100", "H200", "A10G"
GPU_MEMORY_GB = None                       # only for a GPU the planner does not know
MAX_LENGTH = 1536
EPOCHS = 3
RUN_DIR = "turnval/runs/gptoss20b_lora"

# real logs (the hand-labelled test set)
REAL_LOGS = []                             # OTel JSON or span-table exports
REAL_TO_LABEL = "turnval/out/real_to_label.csv"
REAL_LABELLED = ""                         # the same file with a label column filled in (1 / 0)

SMOKE_TEST = os.environ.get("TURNVAL_SMOKE", "0") == "1"


In [ ]:
# ---------------------------------------------------------------- Setup (smoke-test overrides, GPUs, imports)
import sys, json, glob
os.chdir(REPO)
sys.path.insert(0, REPO)

if SMOKE_TEST:                             # no GPU, no server: a stand-in vLLM and a tiny model
    from turnval.mock_vllm import start
    VLLM_URL, BACKEND = start(), "local"
    OUT_DIR, PILOT_DIR, RUN_DIR = "/tmp/turnval_smoke/out", "/tmp/turnval_smoke/pilot", "/tmp/turnval_smoke/run"
    PILOT_LIMIT, FULL_LIMIT, REPLICATES, EPOCHS, MAX_LENGTH, WORKERS = 6, 40, 1, 1, 192, 4
    REAL_LOGS = sorted(glob.glob("/mnt/user-data/uploads/*bot_log*.txt"))[:1]
    REAL_TO_LABEL = "/tmp/turnval_smoke/real_to_label.csv"
    Path("/tmp/turnval_smoke").mkdir(exist_ok=True)

from turnval.train.hardware import select_gpus
if not SMOKE_TEST:
    print("CUDA_VISIBLE_DEVICES =", select_gpus(NUM_GPUS))   # before torch touches CUDA
from turnval import api
print("repo:", REPO, "| smoke test:", SMOKE_TEST, "| text backend:", BACKEND, VLLM_URL if BACKEND == "local" else GATEWAY_MODEL)


## 1. Data: the AOP flow and the HRT graph load, and the walker replays the reference call

In [ ]:
cfg = api.config(backend=BACKEND, vllm_url=VLLM_URL, vllm_model=VLLM_MODEL, gateway_model=GATEWAY_MODEL,
                 out_dir=OUT_DIR, replicates=REPLICATES, workers=WORKERS, verify=VERIFY)
api.check_data(cfg)

## 2. Text model: the vLLM server (or the gateway) answers in JSON

In [ ]:
api.check_llm(cfg)

## 3. Design: journey factors and the covering array

The audit should print `{}`: every outcome the flow branches on is driven by some factor level.

In [ ]:
print("flow outcomes not driven by a factor:", api.audit(cfg))
design_df, design_rep = api.design(cfg)
print({k: design_rep[k] for k in ("factors", "targets", "rows", "uncovered", "conversations")})
design_df.head()

## 4. Pilot: a few conversations to read before generating everything

In [ ]:
pilot = api.generate(cfg, limit=PILOT_LIMIT, out_dir=PILOT_DIR, split=False)
print(json.dumps({k: pilot["report"][k] for k in ("conversations", "failed", "rows", "positives", "negatives",
                                                   "rejected_by_quality_gate")}, indent=1))

In [ ]:
_ = api.show_conversation(pilot, 0)

## 5. Full dataset → `rows.csv` → stratified `train.csv` / `val.csv` / `test.csv`

In [ ]:
data = api.generate(cfg, limit=FULL_LIMIT)
print(json.dumps({k: data["report"][k] for k in ("conversations", "failed", "rows", "positives", "negatives",
                                                  "rejected_by_quality_gate", "seconds")}, indent=1))
print({s: data[s] for s in ("train", "val", "test")})

## 6. Split balance: each factor level's share of train / val / test

In [ ]:
levels, split_rep = api.split_report(data)
print("rows per split:", split_rep["rows"], "| positive rate:", split_rep["positive_rate"],
      "| largest deviation (levels with n >= 10):", split_rep["max_deviation"])
levels.head(15)

## 7. GPUs: size the gpt-oss-20b training to the GPUs you have

In [ ]:
if SMOKE_TEST:
    plan = None
    print("smoke test: tiny model on CPU, no GPU plan")
else:
    plan = api.gpu_plan(NUM_GPUS, GPU_NAME, GPU_MEMORY_GB, max_length=MAX_LENGTH)
    print(json.dumps({k: plan[k] for k in ("num_gpus", "gpu", "memory_gb_per_gpu", "free_for_activations_gb",
                                           "notes", "overrides", "seen")}, indent=1, default=str))

## 8. Training: LoRA, with BCE + pairwise margin + supervised contrastive losses

In [ ]:
overrides = {"train.epochs": EPOCHS}
if SMOKE_TEST:
    overrides.update({"model.backbone": "tiny-random", "model.gradient_checkpointing": False, "model.proj_dim": 32,
                      "data.max_length": MAX_LENGTH, "train.grad_accum": 1})
tcfg = api.train_config(data, plan, out_dir=RUN_DIR, **overrides)
print(json.dumps({"model": tcfg["model"]["backbone"], "data": tcfg["data"], "batch_rows": tcfg["train"]["batch_rows"],
                  "grad_accum": tcfg["train"]["grad_accum"], "loss": tcfg["loss"]}, indent=1, default=str))

In [ ]:
run = api.train(tcfg)
print(json.dumps({"run_dir": run["run_dir"], "thresholds": run["summary"]["thresholds"], "fit": run["summary"]["fit"]},
                 indent=1, default=str))

## 9. Reports: overall, at the 95% thresholds, and per category

In [ ]:
_ = api.show_report(run, "test")

In [ ]:
cats = api.categories(run, "test")
cats[cats.category.str.startswith("factor:")].sort_values("accuracy").head(15)

## 10. Real logs: rows to hand-label, then the labelled real test set

The real logs are summarised by the same summariser as the training data. Fill in the `label` column (1 / 0) of `REAL_TO_LABEL`, save it as `REAL_LABELLED`, then run the last cell.

In [ ]:
if REAL_LOGS:
    api.logs_to_label(REAL_LOGS, REAL_TO_LABEL, cfg)
    import pandas as pd
    display(pd.read_csv(REAL_TO_LABEL, keep_default_na=False)[["conv_id", "turn", "aop", "overlap", "customer", "response"]].head())
else:
    print("no REAL_LOGS given")

In [ ]:
if SMOKE_TEST and REAL_LOGS:                # the smoke test labels every turn 1 just to run the code path
    import pandas as pd
    _df = pd.read_csv(REAL_TO_LABEL, keep_default_na=False); _df["label"] = 1
    REAL_LABELLED = REAL_TO_LABEL.replace(".csv", "_labelled.csv"); _df.to_csv(REAL_LABELLED, index=False)
if REAL_LABELLED:
    real = api.evaluate_labelled(run, REAL_LABELLED)
    print(json.dumps(real["overall@best_f1"], indent=1))
else:
    print("no REAL_LABELLED file yet")